In [ ]:
import os
import re
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
results_dir = "../offline_evaluation_results_gptoss"
results_output_dir = "../analysis_results"

In [ ]:
def load_all_strategies(results_dir="results"):
    """
    Iterates through all JSONL files in the results directory, extracts parameters
    from filenames, loads their contents, and returns a combined DataFrame.
    """
    pattern = re.compile(
        r"strategies_(?P<model>.+?)_(?P<turn>\d+)_top(?P<topk>\d+)logprobs_tail(?P<tail>\d+)_group(?P<group>\d+)\.jsonl"
    )

    all_records = []

    for filename in os.listdir(results_dir):
        if not filename.endswith(".jsonl"):
            continue

        match = pattern.match(filename)
        if not match:
            continue  # skip files that don't fit the pattern

        params = match.groupdict()
        file_path = os.path.join(results_dir, filename)

        # Load JSONL file
        try:
            with open(file_path) as f:
                data_strategies = json.load(f)

            strategy_details = data_strategies.get("strategy_results", {})

            for strategy_name, strategy_data in strategy_details.items():
                task_acc = strategy_data.get("accuracy", None)
                if task_acc is not None:
                    all_records.append({
                        'model': params["model"],
                        "turn": int(params["turn"]),
                        "topk": int(params["topk"]),
                        'n_tail': int(params['tail']),
                        'group_size': int(params['group']),
                        "strategy": strategy_name,
                        "task_accuracy": task_acc,
                    })
        except Exception as e:
            print(f"⚠️ Skipping malformed file: {filename} due to error: {e}")
            continue
        # Add parameters as columns
        
    if not all_records:
        print("No valid JSONL files found.")
        return pd.DataFrame()

    return pd.DataFrame(all_records)

In [ ]:
df_strategies = load_all_strategies(results_dir=f"{results_dir}")


In [ ]:
#df_strategies = df_strategies.loc[df_strategies['turn'].isin([1,2,3])]

In [ ]:
df_strategies.n_tail.unique()

In [ ]:


def plot_strategy_heatmap(df_strategies, group_col=None, title=""):
    """
    Plots either:
      - Standard strategy accuracy heatmap (if group_col=None)
      - Grouped version (rows=metrics, cols=group_col values)
    Also annotates oracle and random strategy baselines (mean ± std).
    """
    model = df_strategies["model"].iloc[0]

    # === 1️⃣ Reshape depending on grouping ===
    def reshape_strategy_accuracies(df_strategies, group_col=None):
        group_keys = ["strategy"] + ([group_col] if group_col else [])
        df_avg = df_strategies.groupby(group_keys, as_index=False)["task_accuracy"].mean()

        def parse_strategy(s):
            m = re.match(r"^(highest|lowest)_(.+)_(full|tail|group_.+)$", s)
            if m:
                return m.group(1), m.group(2), m.group(3)
            return None, None, None

        df_avg[["position", "metric", "subset"]] = pd.DataFrame(
            df_avg["strategy"].apply(parse_strategy).tolist(),
            index=df_avg.index
        )
        df_avg = df_avg.dropna(subset=["position", "metric"])

        if group_col is None:
            pivot_df = (
                df_avg.pivot_table(
                    index="metric",
                    columns=["subset", "position"],
                    values="task_accuracy"
                )
                .sort_index(axis=1, level=[0, 1])
                .round(3)
            )
        else:
            pivot_df = (
                df_avg.pivot_table(
                    index="metric",
                    columns=group_col,
                    values="task_accuracy",
                    aggfunc="mean"
                )
                .sort_index(axis=1)
                .round(3)
            )
        return pivot_df

    result_df = reshape_strategy_accuracies(df_strategies, group_col)
    metric_order = [
            "mean", "median", "variance", "gap", "gap_probs",
            "entropy", 
        ]
    result_df = result_df.reindex(metric_order)


    # === 2️⃣ Compute per-strategy statistics (mean ± std) ===
    stats = (
        df_strategies
        .groupby(["strategy"] + ([group_col] if group_col else []))
        .agg(mean_acc=("task_accuracy", "mean"), std_acc=("task_accuracy", "std"))
        .reset_index()
    )

    # Get baselines
    random_acc = stats[stats["strategy"] == "random"].iloc[0]
    oracle_acc = stats[stats["strategy"] == "oracle"].iloc[0]

    # === 4️⃣ Plot heatmap ===
    data = result_df.values.astype(float)
    fig, ax = plt.subplots(
        figsize=(1.5 * len(result_df.columns), 0.8 * len(result_df.index)), constrained_layout=True

    )
    im = ax.imshow(data, cmap="Blues", aspect="auto", vmin=0.3, vmax=0.8)

    # === 5️⃣ Labels ===
    if group_col:
        xticklabels = [str(c) for c in result_df.columns]
        title_suffix = f"(Grouped by {group_col})"
    else:
        xticklabels = [f"{subset}\n({pos})" for subset, pos in result_df.columns]
        title_suffix = "(Standard Strategy Heatmap)"

    ax.set_xticks(np.arange(len(result_df.columns)))
    ax.set_yticks(np.arange(len(result_df.index)))
    ax.set_xticklabels(xticklabels, rotation=45, ha="right")
    ax.set_yticklabels(result_df.index)
    ax.set_title(f"Strategy Accuracy Heatmap\n{title_suffix} - {model} -\n{title}")

    # === 6️⃣ Annotate cells with mean ± std ===
    for i, metric in enumerate(result_df.index):
        for j, col in enumerate(result_df.columns):
            mean_val = result_df.iloc[i, j]
            if pd.isna(mean_val):
                continue

            if group_col:
                match = stats[(stats[group_col] == col) & (stats["strategy"].str.contains(metric))]
            else:
                subset, pos = col
                match = stats[stats["strategy"] == f"{pos}_{metric}_{subset}"]

            if not match.empty:
                std_val = match["std_acc"].mean()
                ax.text(j, i, f"{mean_val:.2f}±{std_val:.2f}", ha="center", va="center", color="black", fontsize=7)
            else:
                ax.text(j, i, f"{mean_val:.2f}", ha="center", va="center", color="black", fontsize=8)

    # === 7️⃣ Add colorbar ===
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Task Accuracy (mean ± std)", rotation=270, labelpad=15)

    # === 8️⃣ Add oracle and random baselines ===
    y_offset = -0  # space below the heatmap
    text_x = len(result_df.columns)   # centered horizontally

    if not np.isnan(random_acc["mean_acc"]):
        ax.text(
            text_x, len(result_df.index) + y_offset,
            f"Random: {random_acc['mean_acc']:.3f} ± {random_acc['std_acc']:.3f}",
            ha="center", va="center", fontsize=10, color="gray"
        )

    if not np.isnan(oracle_acc["mean_acc"]):
        ax.text(
            text_x, len(result_df.index) + y_offset + 0.6,
            f"Oracle: {oracle_acc['mean_acc']:.3f} ± {oracle_acc['std_acc']:.3f}",
            ha="center", va="center", fontsize=10, color="black", fontweight="bold"
        )

    return fig


In [ ]:
df_strategies_main = df_strategies.loc[(df_strategies.topk==10) & (df_strategies.n_tail==2048) & (df_strategies.group_size==1024) ]

In [ ]:
df_strategies_main

In [ ]:
model = df_strategies_main["model"].iloc[0]

In [ ]:
fig_accuracies_full = plot_strategy_heatmap(df_strategies_main)
fig_accuracies_full.savefig(
    os.path.join(f"{results_output_dir}/figures/{model}/strategy_accuracy_heatmap_full.png"),
    dpi=300
)

In [ ]:
strategies = df_strategies_main["strategy"].unique()
strategies_tail_highest = [s for s in strategies if "tail" in s and "highest" in s]
strategies_group_bottom_pct = [s for s in strategies if "group_bottom_pct" in s and 'highest' in s]
strategies_tail_highest, strategies_group_bottom_pct

In [ ]:
df_strategies_tail = df_strategies.loc[(df_strategies.topk==10) & (df_strategies.group_size==1024) & (df_strategies.strategy.isin(strategies_tail_highest + ['random', 'oracle'])) ]
fig_tail_accuracy = plot_strategy_heatmap(df_strategies_tail, group_col="n_tail", title='tail strategies')
fig_tail_accuracy.savefig(
    os.path.join(f"{results_output_dir}/figures/{model}/strategy_accuracy_heatmap_tail.png"),
    dpi=300
)


In [ ]:
df_strategies_group = df_strategies.loc[(df_strategies.topk==10) & (df_strategies.n_tail==2048) & (df_strategies.strategy.isin(strategies_group_bottom_pct + ['random', 'oracle'])) ]
fig_group_accuracy = plot_strategy_heatmap(df_strategies_group, group_col="group_size", title='group bottom pct strategies')
fig_group_accuracy.savefig(
    os.path.join(f"{results_output_dir}/figures/{model}/strategy_accuracy_heatmap_group.png"),
    dpi=300
)


In [ ]:
df_strategies_topk = df_strategies.loc[(df_strategies.group_size==1024) & (df_strategies.n_tail==2048) & (df_strategies.strategy.isin(strategies_tail_highest + ['random', 'oracle'])) ]
fig_topk_accuracy = plot_strategy_heatmap(df_strategies_topk, group_col="topk", title='tail highest strategies')
fig_topk_accuracy.savefig(
    os.path.join(f"{results_output_dir}/figures/{model}/strategy_accuracy_heatmap_topk.png"),
    dpi=300
)


In [ ]:
### to create the csv below, run big_analysis.ipynb !!!
df_subsampled_computed_accuracies = pd.read_csv(f"{results_dir}/computed_strategies_accuracies.csv")

In [ ]:
df_strategies_topk = df_subsampled_computed_accuracies.loc[(df_subsampled_computed_accuracies.group_size==1024) & (df_subsampled_computed_accuracies.n_tail==2048) & (df_subsampled_computed_accuracies.strategy.isin(strategies_tail_highest + ['random', 'oracle'])) ]
fig_topk_accuracy = plot_strategy_heatmap(df_strategies_topk, group_col="samples_per_group", title='tail highest strategies')
fig_topk_accuracy.savefig(
    os.path.join(f"{results_output_dir}/figures/{model}/strategy_accuracy_heatmap_topk_subsampled.png"),
    dpi=300
)